## Data Quality Issues Found in Bronze Sales Transactions

### Issues Identified:

1. **Null Values (3 records)**
   - Missing `customer_id` in 3 transactions

2. **Invalid Discount Values (1 record)**
   - `discount_pct > 1` (should be between 0 and 1)

3. **Negative Quantity (1 record)**
   - Invalid negative `quantity` value

4. **Duplicate Transaction IDs (5 duplicates)**
   - Same transaction_id appearing multiple times

5. **Inconsistent Order Status Casing (3 variants)**
   - "Completed", "COMPLETE", "completed"
   - " cancelled " (with leading/trailing spaces)

6. **Invalid Date Format (2 records)**
   - Dates that cannot be parsed to DATE type

7. **Wrong Data Type**
   - `order_date` stored as STRING instead of DATE

---

### Cleaning Strategy:

1. Remove duplicate transactions (keep first occurrence)
2. Handle null customer_ids (filter out or replace with placeholder)
3. Cap discount_pct to valid range (0-1)
4. Remove negative quantity records
5. Standardize order_status values (UPPER/LOWER/TRIM)
6. Fix invalid dates (replace with NULL or default)
7. Cast order_date to proper DATE type
8. Add calculated revenue and profit columns

In [0]:
%sql
-- Clean and Transform Sales Transactions from Bronze to Silver
-- This SQL handles all data quality issues identified

WITH cleaned_data AS (
  SELECT 
    -- Original columns with cleaning applied
    transaction_id,
    order_id,
    
    -- Fix date: Convert STRING to DATE, handle invalid dates
    TRY_CAST(order_date AS DATE) as order_date,
    
    -- Handle null customer_id: Replace with 'UNKNOWN'
    COALESCE(customer_id, 'UNKNOWN') as customer_id,
    
    product_id,
    store_id,
    
    -- Ensure positive quantity
    CASE 
      WHEN quantity IS NULL OR quantity < 0 THEN NULL
      ELSE quantity 
    END as quantity,
    
    unit_price,
    unit_cost,
    
    -- Cap discount_pct to valid range [0, 1]
    CASE 
      WHEN discount_pct < 0 THEN 0
      WHEN discount_pct > 1 THEN 1
      ELSE discount_pct
    END as discount_pct,
    
    -- Standardize order_status: Remove spaces, proper case
    INITCAP(TRIM(order_status)) as order_status,
    
    payment_method,
    
    -- Add row number for deduplication
    ROW_NUMBER() OVER (PARTITION BY transaction_id ORDER BY order_date DESC NULLS LAST) as row_num
    
  FROM `end-to-end_pipeline`.bronze.sales_transactions
  
  -- Filter out records with critical data issues
  WHERE product_id IS NOT NULL 
    AND store_id IS NOT NULL
    AND quantity > 0  -- Remove negative or zero quantities
),

deduped_data AS (
  -- Remove duplicates: Keep only the first occurrence
  SELECT 
    transaction_id,
    order_id,
    order_date,
    customer_id,
    product_id,
    store_id,
    quantity,
    unit_price,
    unit_cost,
    discount_pct,
    order_status,
    payment_method
  FROM cleaned_data
  WHERE row_num = 1  -- Keep only first occurrence of each transaction_id
)

-- Final SELECT with calculated columns
SELECT 
  transaction_id,
  order_id,
  order_date,
  customer_id,
  product_id,
  store_id,
  quantity,
  unit_price,
  unit_cost,
  discount_pct,
  order_status,
  payment_method,
  
  -- Calculated columns
  ROUND(quantity * unit_price * (1 - discount_pct), 2) as total_revenue,
  ROUND(quantity * unit_cost, 2) as total_cost,
  ROUND(quantity * unit_price * (1 - discount_pct) - quantity * unit_cost, 2) as profit,
  
  -- Data quality flag
  CASE 
    WHEN customer_id = 'UNKNOWN' THEN 'Missing Customer'
    WHEN order_date IS NULL THEN 'Invalid Date'
    ELSE 'Clean'
  END as data_quality_flag
  
FROM deduped_data

-- Only keep records with valid dates for analysis
WHERE order_date IS NOT NULL

ORDER BY order_date DESC, transaction_id

In [0]:
%sql
-- Create or replace the cleaned Silver table
CREATE OR REPLACE TABLE `end-to-end_pipeline`.silver.sales_transactions_cleaned AS

WITH cleaned_data AS (
  SELECT 
    transaction_id,
    order_id,
    TRY_CAST(order_date AS DATE) as order_date,
    COALESCE(customer_id, 'UNKNOWN') as customer_id,
    product_id,
    store_id,
    CASE 
      WHEN quantity IS NULL OR quantity < 0 THEN NULL
      ELSE quantity 
    END as quantity,
    unit_price,
    unit_cost,
    CASE 
      WHEN discount_pct < 0 THEN 0
      WHEN discount_pct > 1 THEN 1
      ELSE discount_pct
    END as discount_pct,
    INITCAP(TRIM(order_status)) as order_status,
    payment_method,
    ROW_NUMBER() OVER (PARTITION BY transaction_id ORDER BY order_date DESC NULLS LAST) as row_num
  FROM `end-to-end_pipeline`.bronze.sales_transactions
  WHERE product_id IS NOT NULL 
    AND store_id IS NOT NULL
    AND quantity > 0
),

deduped_data AS (
  SELECT 
    transaction_id,
    order_id,
    order_date,
    customer_id,
    product_id,
    store_id,
    quantity,
    unit_price,
    unit_cost,
    discount_pct,
    order_status,
    payment_method
  FROM cleaned_data
  WHERE row_num = 1
)

SELECT 
  transaction_id,
  order_id,
  order_date,
  customer_id,
  product_id,
  store_id,
  quantity,
  unit_price,
  unit_cost,
  discount_pct,
  order_status,
  payment_method,
  ROUND(quantity * unit_price * (1 - discount_pct), 2) as total_revenue,
  ROUND(quantity * unit_cost, 2) as total_cost,
  ROUND(quantity * unit_price * (1 - discount_pct) - quantity * unit_cost, 2) as profit,
  CASE 
    WHEN customer_id = 'UNKNOWN' THEN 'Missing Customer'
    WHEN order_date IS NULL THEN 'Invalid Date'
    ELSE 'Clean'
  END as data_quality_flag,
  CURRENT_TIMESTAMP() as processed_at
FROM deduped_data
WHERE order_date IS NOT NULL

In [0]:
%sql
-- Compare Bronze (before) vs Silver (after) cleaning
SELECT 
  'Bronze (Raw)' as layer,
  COUNT(*) as total_records,
  COUNT(DISTINCT transaction_id) as unique_transactions,
  SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) as null_customer_ids,
  SUM(CASE WHEN quantity < 0 THEN 1 ELSE 0 END) as negative_quantities,
  SUM(CASE WHEN discount_pct > 1 THEN 1 ELSE 0 END) as invalid_discounts,
  COUNT(DISTINCT order_status) as distinct_status_values
FROM `end-to-end_pipeline`.bronze.sales_transactions

UNION ALL

SELECT 
  'Silver (Cleaned)' as layer,
  COUNT(*) as total_records,
  COUNT(DISTINCT transaction_id) as unique_transactions,
  SUM(CASE WHEN customer_id = 'UNKNOWN' THEN 1 ELSE 0 END) as null_customer_ids,
  SUM(CASE WHEN quantity < 0 THEN 1 ELSE 0 END) as negative_quantities,
  SUM(CASE WHEN discount_pct > 1 THEN 1 ELSE 0 END) as invalid_discounts,
  COUNT(DISTINCT order_status) as distinct_status_values
FROM `end-to-end_pipeline`.silver.sales_transactions_cleaned